# iSTS results

Tables from one evaluation run, following section 8 of the [Jacyna report](../materials/Jacyna/Nlp_P_Pusiewicz_Z_Jacyna_M_Rak_uzupelnione.pdf). Set `RUN_DIR` to the folder that contains the evaluator outputs.

In [1]:
import csv
import json
from pathlib import Path

from IPython.display import Markdown, display

REPO_DIR = Path.cwd()
if REPO_DIR.name == "notebooks":
    REPO_DIR = REPO_DIR.parent

RUN_DIR = REPO_DIR / "benchmark_results/gpt-6.1-sol-system"

single_csv = RUN_DIR / "results.csv"
result_files = [single_csv] if single_csv.is_file() else sorted(RUN_DIR.glob("*/results.csv"))
if not result_files:
    raise FileNotFoundError(f"No results.csv files under {RUN_DIR.resolve()}")

results = {}
result_dirs = {}
for path in result_files:
    manifest = path.with_name("run.json")
    prompt = path.parent.name
    if manifest.is_file():
        prompt = Path(json.loads(manifest.read_text(encoding="utf-8"))["prompt"]).stem

    result_dirs[prompt] = path.parent
    with path.open(encoding="utf-8", newline="") as file:
        results[prompt] = {row["domain"]: row for row in csv.DictReader(file)}

rows = [row for domain_rows in results.values() for row in domain_rows.values()]
models = {row["model"] for row in rows}
chunk_modes = {row["chunks"] for row in rows}
if len(models) != 1 or len(chunk_modes) != 1:
    raise ValueError("Choose a run with scored results for one model and one chunk mode.")

prompt_order = (
    "zero-shot.ists", "one-shot.ists",
    "zero-shot.jacyna-json", "one-shot.jacyna-json",
    "zero-shot.json", "one-shot.json", "eight-shot.ists",
)
prompts = [prompt for prompt in prompt_order if prompt in results]
prompts.extend(sorted(results.keys() - set(prompts)))

model = models.pop()
chunk_mode = {"system": "SystemChunks", "golden": "GoldChunks"}[chunk_modes.pop()]
print(f"{model} · {chunk_mode} · {len(prompts)} prompts")

gpt-6.1-sol · SystemChunks · 7 prompts


## Scores by domain

Each row uses the scores saved by the evaluator. A dash marks an unavailable score.

In [2]:
def show_table(title, columns, rows):
    """Display a Markdown table with numeric columns aligned to the right."""
    rules = ["---", *["---:"] * (len(columns) - 1)]
    lines = [
        "| " + " | ".join(str(value).replace("|", r"\|") for value in row) + " |"
        for row in [columns, rules, *rows]
    ]
    display(Markdown(f"### {title}\n\n" + "\n".join(lines)))


metrics = {
    "f1_score": "F1 Score",
    "f1_type": "F1 Type",
    "f1_type_score_no_penalty": "F1 Type + Score (no penalty)",
    "f1_type_score_penalty": "F1 Type + Score (penalty)",
}
domains = {"answers-students": "Answers", "images": "Images", "headlines": "Headlines"}

for domain, title in domains.items():
    table = []
    for prompt in prompts:
        row = results[prompt].get(domain)
        if row:
            table.append([prompt, *(row[metric] or "—" for metric in metrics)])

    if table:
        show_table(title, ["Prompt", *metrics.values()], table)

### Answers

| Prompt | F1 Score | F1 Type | F1 Type + Score (no penalty) | F1 Type + Score (penalty) |
| --- | ---: | ---: | ---: | ---: |
| zero-shot.ists | 0.6390 | 0.4949 | 0.4822 | 0.5444 |
| one-shot.ists | 0.6337 | 0.4891 | 0.4765 | 0.5382 |
| zero-shot.jacyna-json | 0.7245 | 0.5536 | 0.5405 | 0.6051 |
| one-shot.jacyna-json | 0.7271 | 0.5756 | 0.5607 | 0.6220 |
| zero-shot.json | 0.7093 | 0.5334 | 0.5193 | 0.5938 |
| one-shot.json | 0.7159 | 0.5625 | 0.5474 | 0.6104 |
| eight-shot.ists | 0.7997 | 0.6614 | 0.6459 | 0.6892 |

### Images

| Prompt | F1 Score | F1 Type | F1 Type + Score (no penalty) | F1 Type + Score (penalty) |
| --- | ---: | ---: | ---: | ---: |
| zero-shot.ists | 0.8395 | 0.6568 | 0.6430 | 0.7249 |
| one-shot.ists | 0.8264 | 0.6337 | 0.6218 | 0.7170 |
| zero-shot.jacyna-json | 0.8342 | 0.6513 | 0.6396 | 0.7205 |
| one-shot.jacyna-json | 0.8351 | 0.6465 | 0.6341 | 0.7233 |
| zero-shot.json | 0.8332 | 0.6414 | 0.6305 | 0.7170 |
| one-shot.json | 0.8259 | 0.6299 | 0.6184 | 0.7116 |
| eight-shot.ists | 0.8411 | 0.6499 | 0.6381 | 0.7290 |

### Headlines

| Prompt | F1 Score | F1 Type | F1 Type + Score (no penalty) | F1 Type + Score (penalty) |
| --- | ---: | ---: | ---: | ---: |
| zero-shot.ists | 0.7805 | 0.6248 | 0.6111 | 0.6844 |
| one-shot.ists | 0.7698 | 0.6243 | 0.6116 | 0.6775 |
| zero-shot.jacyna-json | 0.7764 | 0.6238 | 0.6112 | 0.6798 |
| one-shot.jacyna-json | 0.7810 | 0.6268 | 0.6146 | 0.6806 |
| zero-shot.json | 0.7820 | 0.6286 | 0.6142 | 0.6888 |
| one-shot.json | 0.7735 | 0.6195 | 0.6072 | 0.6772 |
| eight-shot.ists | 0.7860 | 0.6396 | 0.6258 | 0.6916 |

## All datasets combined

For each prompt with results in all three domains, the Perl evaluators score the saved predictions and their references together. The pair count reflects the evaluated subset. This table requires Perl and the saved `.wa` files.

In [3]:
import re
import subprocess
from tempfile import TemporaryDirectory

sentence_record = re.compile(r'^<sentence id="(\d+)"[^>]*>.*?</sentence>', re.M | re.S)
sentence_header = re.compile(r'^<sentence id="\d+"')


def wa_records(path):
    """Read the .wa records in file order, keyed by sentence ID."""
    return {
        match[1]: match[0]
        for match in sentence_record.finditer(path.read_text(encoding="utf-8"))
    }


def scorer_f1(script, gold_path, predictions_path):
    """Read the F1 measures printed by one Perl scorer."""
    result = subprocess.run(
        ["perl", str(script), "--", str(gold_path), str(predictions_path)],
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(f"{script.name}: {result.stderr.strip()}")
    return dict(re.findall(r"^\s*F1 (Score|Type|Typ\+Sco)\s+(\d+\.\d+)\s*$", result.stdout, re.M))


complete_prompts = [prompt for prompt in prompts if domains.keys() <= results[prompt].keys()]
if not complete_prompts:
    print("No prompt has results for all three domains yet.")
else:
    references = {
        domain: wa_records(REPO_DIR / "data/test_goldStandard" / f"STSint.testinput.{domain}.wa")
        for domain in domains
    }
    table = []
    with TemporaryDirectory() as directory:
        gold_path = Path(directory) / "gold.wa"
        predictions_path = Path(directory) / "predictions.wa"
        for prompt in complete_prompts:
            pooled_gold = []
            pooled_predictions = []
            for domain in domains:
                predictions = wa_records(result_dirs[prompt] / domain / "predictions.wa")
                if len(predictions) != int(results[prompt][domain]["pairs"]):
                    raise ValueError(f"Saved prediction count differs from results.csv for {prompt} / {domain}.")
                for pair_id, prediction in predictions.items():
                    header = f'<sentence id="{len(pooled_gold) + 1}"'
                    pooled_gold.append(sentence_header.sub(header, references[domain][pair_id], count=1))
                    pooled_predictions.append(sentence_header.sub(header, prediction, count=1))

            gold_path.write_text("\n".join(pooled_gold) + "\n", encoding="utf-8")
            predictions_path.write_text("\n".join(pooled_predictions) + "\n", encoding="utf-8")
            standard = scorer_f1(REPO_DIR / "scripts/evalF1_no_penalty.pl", gold_path, predictions_path)
            penalty = scorer_f1(REPO_DIR / "scripts/evalF1_penalty.pl", gold_path, predictions_path)
            table.append([
                prompt, len(pooled_gold), standard["Score"], standard["Type"],
                standard["Typ+Sco"], penalty["Typ+Sco"],
            ])

    show_table("Pooled scores", ["Prompt", "Pairs", *metrics.values()], table)

### Pooled scores

| Prompt | Pairs | F1 Score | F1 Type | F1 Type + Score (no penalty) | F1 Type + Score (penalty) |
| --- | ---: | ---: | ---: | ---: | ---: |
| zero-shot.ists | 1094 | 0.7585 | 0.5962 | 0.5828 | 0.6558 |
| one-shot.ists | 1094 | 0.7484 | 0.5855 | 0.5731 | 0.6486 |
| zero-shot.jacyna-json | 1094 | 0.7801 | 0.6101 | 0.5977 | 0.6695 |
| one-shot.jacyna-json | 1094 | 0.7827 | 0.6168 | 0.6036 | 0.6765 |
| zero-shot.json | 1094 | 0.7761 | 0.6010 | 0.5880 | 0.6669 |
| one-shot.json | 1094 | 0.7730 | 0.6039 | 0.5910 | 0.6670 |
| eight-shot.ists | 1094 | 0.8109 | 0.6508 | 0.6372 | 0.7045 |

## Reply conversion failures

Counts are summed across scored domains. Rejected replies are the failures left after any repairs. A dash in Malformed JSON applies to iSTS text answers.

In [4]:
table = []
for prompt in prompts:
    domain_rows = results[prompt].values()
    replies = sum(int(row["pairs"]) for row in domain_rows)
    rejected = sum(int(row["invalid"]) for row in domain_rows)
    repaired = sum(int(row["repaired"]) for row in domain_rows)
    malformed = [int(row["malformed_json"]) for row in domain_rows if row["malformed_json"]]
    table.append([
        prompt, replies, rejected, f"{rejected / replies:.2%}",
        sum(malformed) if malformed else "—", repaired,
    ])

show_table(
    "All scored domains",
    ["Prompt", "Replies", "Rejected", "Rejected (%)", "Malformed JSON", "Repaired"],
    table,
)

### All scored domains

| Prompt | Replies | Rejected | Rejected (%) | Malformed JSON | Repaired |
| --- | ---: | ---: | ---: | ---: | ---: |
| zero-shot.ists | 1094 | 63 | 5.76% | — | 0 |
| one-shot.ists | 1094 | 63 | 5.76% | — | 0 |
| zero-shot.jacyna-json | 1094 | 0 | 0.00% | 0 | 0 |
| one-shot.jacyna-json | 1094 | 0 | 0.00% | 0 | 0 |
| zero-shot.json | 1094 | 0 | 0.00% | 0 | 0 |
| one-shot.json | 1094 | 0 | 0.00% | 0 | 0 |
| eight-shot.ists | 1094 | 0 | 0.00% | — | 0 |